# CS521 HW3 — Problem 3: Multi-Norm Adversarial Training

**Run in Colab with GPU:** Runtime → Change runtime type → T4 GPU.

## 1. Setup and download data

In [ ]:
!pip -q install gdown
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 28.0MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=970f870f-e291-44a0-ba8e-fd5c407c4a06
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 77.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 17.1MB/s]


In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [ ]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [ ]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

In [ ]:
from pathlib import Path
import json
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CS521_HW3')
MODELS = ROOT / 'models'
RESULTS = ROOT / 'results'
MODELS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print('Saving new runs to:', ROOT)


Mounted at /content/drive
Saving new runs to: /content/drive/MyDrive/CS521_HW3


## 2. PGD attacks and evaluation

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1].
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):

    # Start with random perturbation
    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)
    else:
        delta = torch.zeros_like(x)

    delta = clip_box(x, delta)

    # PGD attack
    for _ in range(steps):

        # Calculate gradient
        grad = input_grad(model, x, delta, y)

        # Update perturbation
        delta = delta + alpha * grad.sign()

        # Keep perturbation inside Linf ball
        delta = delta.clamp(-eps, eps)

        # Keep image values between 0 and 1
        delta = clip_box(x, delta)

    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):

    # Start with random perturbation inside L2 ball
    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp_min(1e-12)

        radius = torch.rand(
            x.shape[0], 1, 1, 1, device=x.device
        ) ** (1.0 / x[0].numel())

        delta = delta * eps * radius
    else:
        delta = torch.zeros_like(x)

    delta = clip_box(x, delta)

    # PGD attack
    for _ in range(steps):

        # Calculate gradient
        grad = input_grad(model, x, delta, y)

        # Normalize gradient
        grad = grad / l2_norm(grad).clamp_min(1e-12)

        # Update perturbation
        delta = delta + alpha * grad

        # Project perturbation into L2 ball
        norm = l2_norm(delta).clamp_min(1e-12)
        factor = (eps / norm).clamp(max=1.0)
        delta = delta * factor

        # Keep image values between 0 and 1
        delta = clip_box(x, delta)

    return (x + delta).detach()

In [ ]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

In [ ]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

## 3. Training function



In [ ]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    history = []
    for epoch in range(epochs):
        epoch_start = time.time()
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        row = {"epoch": epoch + 1, "loss": float(np.mean(losses)), "seconds": round(time.time() - epoch_start, 2), "cumulative_seconds": round(time.time() - start, 2)}
        history.append(row)
        print(f"epoch {epoch + 1}: loss {row['loss']:.3f}, epoch time {row['seconds']:.1f} s, total {row['cumulative_seconds']:.1f} s")
    model.eval()
    model.training_history = history
    return model

In [ ]:
def run_experiment(name, loss_fn, seed=0, gp_beta=None, evaluate=True, **kwargs):
    """Train from the base weights, save checkpoint, timings and measured accuracy."""
    print(f'\n=== {name} ===')
    model = adv_train(load_base_model(), loss_fn, seed=seed, gp_beta=gp_beta, **kwargs)
    torch.save(model.state_dict(), MODELS / f'{name}.pth')
    accuracy = test_model(model) if evaluate else None
    record = {'name': name, 'seed': seed, 'epochs': model.training_history,
              'total_training_seconds': model.training_history[-1]['cumulative_seconds'],
              'accuracy': accuracy, 'evaluation': {'steps': 50, 'restarts': 2} if evaluate else None}
    (RESULTS / f'{name}_run.json').write_text(json.dumps(record, indent=2))
    print('Saved model and results for', name)
    return model, record


## Part (a) — Single-norm adversarial training

In [ ]:
def single_norm_loss(model, x, y, norm):

    # Step 1: Switch to evaluation mode
    model.eval()

    # Step 2: Generate adversarial images
    x_adv = attack(model, x, y, norm)

    # Step 3: Switch back to training mode
    model.train()

    # Step 4: Calculate cross-entropy loss
    loss = F.cross_entropy(model(x_adv), y)

    return loss

In [ ]:
model_linf, result_linf = run_experiment('linf', single_norm_loss, norm='linf')
model_l2, result_l2 = run_experiment('l2', single_norm_loss, norm='l2')
model_l1, result_l1 = run_experiment('l1', single_norm_loss, norm='l1')


=== linf ===
epoch 1: loss 1.203, epoch time 38.1 s, total 38.1 s
epoch 2: loss 1.181, epoch time 38.5 s, total 76.6 s
epoch 3: loss 1.152, epoch time 39.0 s, total 115.7 s
epoch 4: loss 1.126, epoch time 39.4 s, total 155.1 s
epoch 5: loss 1.097, epoch time 39.7 s, total 194.7 s
epoch 6: loss 1.068, epoch time 39.8 s, total 234.6 s
epoch 7: loss 1.049, epoch time 39.9 s, total 274.5 s
epoch 8: loss 1.030, epoch time 40.0 s, total 314.5 s
epoch 9: loss 1.013, epoch time 40.1 s, total 354.6 s
epoch 10: loss 1.008, epoch time 40.2 s, total 394.8 s
clean 78.5  linf 45.1  l2 57.4  l1 20.5  union 20.4   (120 s)
Saved model and results for linf

=== l2 ===
epoch 1: loss 0.892, epoch time 40.3 s, total 40.3 s
epoch 2: loss 0.820, epoch time 40.4 s, total 80.7 s
epoch 3: loss 0.781, epoch time 40.4 s, total 121.1 s
epoch 4: loss 0.748, epoch time 40.4 s, total 161.5 s
epoch 5: loss 0.708, epoch time 40.4 s, total 201.9 s
epoch 6: loss 0.668, epoch time 40.4 s, total 242.4 s
epoch 7: loss 0.62

## Part (b) — RAMP training, two seeds

In [ ]:
def ramp_loss(model, x, y, lam=0.5):

    # Generate adversarial images in evaluation mode
    model.eval()

    x_inf = attack(model, x, y, "linf")
    x_l1 = attack(model, x, y, "l1")

    # Switch back to training mode
    model.train()

    # Predictions for both attacks
    logits_inf = model(x_inf)
    logits_l1 = model(x_l1)

    # Cross-entropy loss for each image
    loss_inf = F.cross_entropy(
        logits_inf, y, reduction="none"
    )

    loss_l1 = F.cross_entropy(
        logits_l1, y, reduction="none"
    )

    # Use the larger loss for each image
    adv_loss = torch.maximum(loss_inf, loss_l1).mean()

    # Select images correctly classified under Linf attack
    correct = logits_inf.argmax(dim=1) == y

    # KL divergence on correctly classified images
    if correct.any():
        log_p_inf = F.log_softmax(logits_inf[correct], dim=1)
        p_l1 = F.softmax(logits_l1[correct], dim=1)

        kl_loss = F.kl_div(
            log_p_inf, p_l1, reduction="batchmean"
        )
    else:
        kl_loss = logits_inf.sum() * 0.0

    return adv_loss + lam * kl_loss


def gradient_projection(g_n, g_a):

    # Flatten gradients
    gn = g_n.flatten()
    ga = g_a.flatten()

    # Calculate cosine similarity
    cosine = F.cosine_similarity(
        gn.unsqueeze(0),
        ga.unsqueeze(0),
        dim=1,
        eps=1e-12
    ).squeeze(0)

    # Only keep natural gradient when aligned
    return torch.clamp(cosine, min=0.0) * g_n

In [ ]:
model_ramp_seed0, result_ramp_seed0 = run_experiment('ramp_seed0', ramp_loss, seed=0, gp_beta=0.5)
model_ramp_seed1, result_ramp_seed1 = run_experiment('ramp_seed1', ramp_loss, seed=1, gp_beta=0.5)


=== ramp_seed0 ===
epoch 1: loss 1.434, epoch time 87.4 s, total 87.4 s
epoch 2: loss 1.331, epoch time 87.0 s, total 174.5 s
epoch 3: loss 1.304, epoch time 86.9 s, total 261.4 s
epoch 4: loss 1.281, epoch time 87.0 s, total 348.4 s
epoch 5: loss 1.254, epoch time 86.9 s, total 435.2 s
epoch 6: loss 1.227, epoch time 87.1 s, total 522.4 s
epoch 7: loss 1.199, epoch time 87.1 s, total 609.5 s
epoch 8: loss 1.186, epoch time 87.0 s, total 696.5 s
epoch 9: loss 1.172, epoch time 87.0 s, total 783.5 s
epoch 10: loss 1.166, epoch time 87.1 s, total 870.6 s
clean 79.1  linf 41.1  l2 62.0  l1 44.6  union 39.1   (121 s)
Saved model and results for ramp_seed0

=== ramp_seed1 ===
epoch 1: loss 1.436, epoch time 87.0 s, total 87.0 s
epoch 2: loss 1.333, epoch time 87.1 s, total 174.2 s
epoch 3: loss 1.299, epoch time 87.1 s, total 261.3 s
epoch 4: loss 1.274, epoch time 87.2 s, total 348.5 s
epoch 5: loss 1.248, epoch time 87.3 s, total 435.8 s
epoch 6: loss 1.222, epoch time 87.1 s, total 522.

## Part (c) — My method: Adaptive Norm Sampling (ANS)

For each batch, select one norm using probabilities based on a moving average of adversarial loss. Higher-loss norms are sampled more often. `my_loss` is an instance of the callable class below, compatible with `adv_train`.

In [ ]:
class AdaptiveNormLoss:
    def __init__(self, rho=0.1, temperature=0.5):
        self.norms = ("linf", "l2", "l1")
        self.rho = rho
        self.temperature = temperature

        # Start with equal loss estimates
        self.running_losses = torch.ones(3)

        # Record how often each norm is selected
        self.counts = torch.zeros(3, dtype=torch.long)

    def __call__(self, model, x, y):

        # Convert recent losses into sampling probabilities
        probabilities = torch.softmax(
            self.running_losses / self.temperature,
            dim=0
        )

        # Choose one attack norm
        selected = torch.multinomial(
            probabilities, num_samples=1
        ).item()

        norm = self.norms[selected]
        self.counts[selected] += 1

        # Generate adversarial images
        model.eval()
        x_adv = attack(model, x, y, norm)

        # Calculate adversarial training loss
        model.train()
        loss = F.cross_entropy(model(x_adv), y)

        # Update the running loss estimate
        with torch.no_grad():
            value = loss.detach().item()
            self.running_losses[selected] = (
                (1 - self.rho) * self.running_losses[selected]
                + self.rho * value
            )

        return loss


In [ ]:
my_loss = AdaptiveNormLoss(rho=0.1, temperature=0.5)
model_mine, result_ans_seed0 = run_experiment('ans_seed0', my_loss, seed=0)
print('ANS seed 0 norm counts:', my_loss.counts.tolist())

my_loss_seed1 = AdaptiveNormLoss(rho=0.1, temperature=0.5)
model_mine_seed1, result_ans_seed1 = run_experiment('ans_seed1', my_loss_seed1, seed=1)
print('ANS seed 1 norm counts:', my_loss_seed1.counts.tolist())


=== ans_seed0 ===
epoch 1: loss 1.250, epoch time 41.5 s, total 41.5 s
epoch 2: loss 1.193, epoch time 39.0 s, total 80.4 s
epoch 3: loss 1.121, epoch time 39.2 s, total 119.7 s
epoch 4: loss 1.087, epoch time 39.7 s, total 159.3 s
epoch 5: loss 1.086, epoch time 39.9 s, total 199.2 s
epoch 6: loss 1.040, epoch time 40.2 s, total 239.4 s
epoch 7: loss 1.024, epoch time 40.2 s, total 279.6 s
epoch 8: loss 1.006, epoch time 40.3 s, total 319.9 s
epoch 9: loss 0.971, epoch time 40.4 s, total 360.3 s
epoch 10: loss 0.942, epoch time 40.5 s, total 400.9 s
clean 78.7  linf 40.9  l2 61.8  l1 45.7  union 39.2   (120 s)
Saved model and results for ans_seed0
ANS seed 0 norm counts: [334, 171, 275]

=== ans_seed1 ===
epoch 1: loss 1.218, epoch time 40.5 s, total 40.5 s
epoch 2: loss 1.173, epoch time 40.6 s, total 81.2 s
epoch 3: loss 1.148, epoch time 40.6 s, total 121.8 s
epoch 4: loss 1.077, epoch time 40.6 s, total 162.5 s
epoch 5: loss 1.082, epoch time 40.6 s, total 203.0 s
epoch 6: loss 1

### Ablation — Uniform sampling

In [ ]:
class UniformNormLoss:
    def __init__(self):
        self.norms = ("linf", "l2", "l1")
        self.counts = torch.zeros(3, dtype=torch.long)

    def __call__(self, model, x, y):

        # Choose each norm with equal probability
        selected = torch.randint(0, 3, (1,)).item()

        norm = self.norms[selected]
        self.counts[selected] += 1

        # Generate adversarial examples
        model.eval()
        x_adv = attack(model, x, y, norm)

        # Calculate training loss
        model.train()
        loss = F.cross_entropy(model(x_adv), y)

        return loss

In [ ]:
uniform_loss = UniformNormLoss()
model_uniform, result_uniform = run_experiment('uniform_seed0', uniform_loss, seed=0)
print('Uniform norm counts:', uniform_loss.counts.tolist())


=== uniform_seed0 ===
epoch 1: loss 1.166, epoch time 39.8 s, total 39.8 s
epoch 2: loss 1.118, epoch time 40.1 s, total 79.9 s
epoch 3: loss 1.052, epoch time 40.2 s, total 120.1 s
epoch 4: loss 1.026, epoch time 40.3 s, total 160.4 s
epoch 5: loss 1.025, epoch time 40.4 s, total 200.9 s
epoch 6: loss 0.973, epoch time 40.5 s, total 241.3 s
epoch 7: loss 0.969, epoch time 40.5 s, total 281.8 s
epoch 8: loss 0.976, epoch time 40.4 s, total 322.2 s
epoch 9: loss 0.925, epoch time 40.6 s, total 362.8 s
epoch 10: loss 0.898, epoch time 40.6 s, total 403.4 s
clean 78.6  linf 40.4  l2 62.9  l1 46.6  union 38.6   (120 s)
Saved model and results for uniform_seed0
Uniform norm counts: [239, 292, 249]


### Stronger attack evaluation (200 steps, 5 restarts)

Run after training, or load checkpoints from Drive using the helper below. This evaluation can take a long time.

In [ ]:
def load_trained_model(name):
    path = MODELS / f'{name}.pth'
    if not path.is_file():
        raise FileNotFoundError(f'Missing checkpoint: {path}. Train this model first.')
    model = make_model()
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    model.eval()
    return model

def strong_evaluation(name):
    model = load_trained_model(name)
    acc = test_model(model, steps=200, restarts=5)
    data = {'name':name,'steps':200,'restarts':5,'accuracy':acc}
    (RESULTS / f'{name}_strong_run.json').write_text(json.dumps(data,indent=2))
    return acc

# Uncomment to run the expensive stronger evaluation:
# strong_ans = strong_evaluation('ans_seed0')
# strong_ramp = strong_evaluation('ramp_seed1')

In [ ]:
# Part (c)(vi): Stronger attack evaluation
# 200 PGD steps and 5 restarts

strong_ans = strong_evaluation('ans_seed0')
strong_ramp = strong_evaluation('ramp_seed1')

print("Stronger evaluation completed.")

clean 78.7  linf 40.9  l2 61.8  l1 44.8  union 38.7   (1200 s)
clean 79.2  linf 41.7  l2 62.6  l1 44.8  union 39.3   (1199 s)
Stronger evaluation completed.


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/CS521_HW3")

for name in ["linf", "l2", "l1", "ramp_seed0", "ramp_seed1"]:
    model = root / "models" / f"{name}.pth"
    result = root / "results" / f"{name}_run.json"

    print(
        name,
        "| Model:", model.exists(),
        "| Results:", result.exists()
    )

linf | Model: True | Results: True
l2 | Model: True | Results: True
l1 | Model: True | Results: True
ramp_seed0 | Model: True | Results: True
ramp_seed1 | Model: True | Results: True
